<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/week_7_adaptation/week7_adaptation_real_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 (real data): Banking77 intent routing, zero-shot vs dynamic few-shot

**Task:** route a customer's banking message to the right intent. This is the same experiment as
my synthetic support-ticket notebook, run on real customer text. That notebook ended at its
ceiling (zero-shot 98.4%, few-shot 100%) and never produced a slice where few-shot lost, which
is part of why I wanted real data.

**Data:** [Banking77](https://github.com/PolyAI-LDN/task-specific-datasets) (PolyAI, Casanueva et
al., 2020): real online-banking customer queries, CC BY 4.0. I download the authors' CSV files
directly. I use **10 of the 77 intents, picked by me and not at random**: four small groups of
look-alike intents (card delivery, card not working, pending payments, declined payments). With
all 77 labels, 100 eval tickets would hit each label about once and no slice would mean anything.
That choice makes the task harder than average, and I'm saying so up front.

**Model:** I used Claude (`claude-haiku-4-5-20251001`)
instead, the same swap already disclosed in Weeks 5-7. Calls pin `temperature` to 0 (the run
prints a note if the API refuses that), and each (variant, ticket) pair is called exactly once and
cached.

The cells were checked against fakes first (a fake embedder, a fake Claude client, and fake CSV
files with known defects). That only proves the plumbing. Every number and every claim about what
the model does comes from the Colab run.


In [1]:
%pip install -q tiktoken sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.5 MB/s eta 0:00:00


## Part 1: Prepare the dataset

I pool the rows for my 10 intents from Banking77's own train and test files and clean them
**before** sampling, so the duplicates and overlaps I report are real ones, not injected. If the
same text appears in both files, the train copy is dropped so no eval ticket is also a training
example. Then I sample a fixed number per intent: 30 for train, 12 for eval, with eval drawn only
from Banking77's test file.


In [2]:
import os, re, io, csv, json, random, hashlib, pathlib, time, math, itertools, urllib.request
from collections import Counter, defaultdict

SEED = 7
rng = random.Random(SEED)
BASE_URL = 'https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/'
RAW_DIR = pathlib.Path('data_raw'); RAW_DIR.mkdir(exist_ok=True)

CLUSTERS = {
    'delivery':    ['card_arrival', 'card_delivery_estimate'],
    'not_working': ['card_not_working', 'virtual_card_not_working', 'contactless_not_working'],
    'pending':     ['pending_card_payment', 'pending_transfer', 'pending_top_up'],
    'declined':    ['declined_card_payment', 'declined_transfer'],
}
LABELS = [l for ls in CLUSTERS.values() for l in ls]
CLUSTER_OF = {l: c for c, ls in CLUSTERS.items() for l in ls}
VALID = set(LABELS)
MAX_CHARS = 300
TRAIN_PER_INTENT = 30
EVAL_PER_INTENT = 12

def load_split(name):
    path = RAW_DIR / name
    if not path.exists():
        try:
            with urllib.request.urlopen(BASE_URL + name, timeout=30) as resp:
                path.write_bytes(resp.read())
        except Exception as e:
            raise RuntimeError(f'Could not download {name} ({type(e).__name__}: {e}). Download train.csv and test.csv '
                               f'from {BASE_URL} and upload them into the data_raw folder, then re-run.') from e
    raw_bytes = path.read_bytes()
    rows = list(csv.DictReader(io.StringIO(raw_bytes.decode('utf-8-sig'))))
    return rows, hashlib.sha256(raw_bytes).hexdigest()[:16]

raw_train_all, HASH_TRAIN = load_split('train.csv')
raw_test_all, HASH_TEST = load_split('test.csv')
file_labels = sorted({(r.get('category') or '').strip().lower() for r in raw_train_all + raw_test_all})
missing = [l for l in LABELS if l not in file_labels]
if missing:
    raise RuntimeError(f'These intents are not in the downloaded files: {missing}. Intents found: {file_labels}')
print(f'downloaded train.csv: {len(raw_train_all)} rows (sha256 {HASH_TRAIN}), test.csv: {len(raw_test_all)} rows (sha256 {HASH_TEST})')
print(f'intents in the files: {len(file_labels)}   intents I use: {len(LABELS)}')

def subset(rows, split):
    return [{'text': r.get('text'), 'label': r.get('category'), 'split': split}
            for r in rows if (r.get('category') or '').strip().lower() in VALID]
raw = subset(raw_train_all, 'train') + subset(raw_test_all, 'test')
RAW_COUNTS = {s: Counter((r['label'] or '').strip().lower() for r in raw if r['split'] == s) for s in ('train', 'test')}
print(f'rows for my {len(LABELS)} intents: {len(raw)}  (train file {sum(RAW_COUNTS["train"].values())}, test file {sum(RAW_COUNTS["test"].values())})')
print('per intent (train/test):', {l: f'{RAW_COUNTS["train"][l]}/{RAW_COUNTS["test"][l]}' for l in LABELS})

downloaded train.csv: 10003 rows (sha256 b06e26ac67551395), test.csv: 3080 rows (sha256 d12d6e3bc4c31039)
intents in the files: 77   intents I use: 10
rows for my 10 intents: 1595  (train file 1195, test file 400)
per intent (train/test): {'card_arrival': '153/40', 'card_delivery_estimate': '112/40', 'card_not_working': '112/40', 'virtual_card_not_working': '41/40', 'contactless_not_working': '35/40', 'pending_card_payment': '159/40', 'pending_transfer': '148/40', 'pending_top_up': '149/40', 'declined_card_payment': '153/40', 'declined_transfer': '133/40'}


In [3]:
def norm_key(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^\w\s]', '', text.lower())).strip()

def clean(records):
    rep = Counter()
    kept = []
    for r in records:
        r = dict(r)
        t = re.sub(r'\s+', ' ', r['text']).strip() if isinstance(r['text'], str) else ''
        l = r['label'].strip().lower() if isinstance(r['label'], str) else ''
        if not t:
            rep['dropped: empty text'] += 1; continue
        if l not in VALID:
            rep['dropped: label not in my intent set'] += 1; continue
        if len(t) > MAX_CHARS:
            rep[f'dropped: longer than {MAX_CHARS} chars'] += 1; continue
        if t != r['text']:
            rep['repaired: whitespace'] += 1
        if l != r['label']:
            rep['repaired: label case/whitespace'] += 1
        r['text'], r['label'] = t, l
        kept.append(r)
    groups = defaultdict(list)
    for r in kept:
        groups[norm_key(r['text'])].append(r)
    out = []
    for grp in groups.values():
        if len({g['label'] for g in grp}) > 1:
            rep['dropped: same text, conflicting labels (all copies)'] += len(grp); continue
        test_members = [g for g in grp if g['split'] == 'test']
        keep = test_members[0] if test_members else grp[0]
        leak = sum(1 for g in grp if g['split'] == 'train') if test_members else 0
        rep['dropped: train copy of a test text (leakage)'] += leak
        rep['dropped: exact/near duplicate'] += len(grp) - 1 - leak
        out.append(keep)
    return out, rep

clean_records, CLEAN_REPORT = clean(raw)
pool_train = [r for r in clean_records if r['split'] == 'train']
pool_test = [r for r in clean_records if r['split'] == 'test']
print(f'raw {len(raw)} -> clean {len(clean_records)}   (train pool {len(pool_train)}, test pool {len(pool_test)})')
for k, v in sorted(CLEAN_REPORT.items()):
    print(f'  {k:55s} {v}')

def sample_per_intent(records, n, what):
    out = []
    for lab in LABELS:
        pool = sorted([r for r in records if r['label'] == lab], key=lambda r: norm_key(r['text']))
        if len(pool) < n:
            print(f'WARNING: only {len(pool)} {what} examples for {lab} (wanted {n})')
        out += rng.sample(pool, min(n, len(pool)))
    return out

train = sample_per_intent(pool_train, TRAIN_PER_INTENT, 'train')
eval_ = sample_per_intent(pool_test, EVAL_PER_INTENT, 'eval')
for r in train + eval_:
    r['cluster'] = CLUSTER_OF[r['label']]
rng.shuffle(train); rng.shuffle(eval_)

overlap = {norm_key(r['text']) for r in train} & {norm_key(r['text']) for r in eval_}
print(f'\nsampled: train {len(train)}, eval {len(eval_)};  train/eval overlap (normalized text): {len(overlap)}')
assert not overlap, 'train and eval share text'
print('requirement >=200 train:', 'PASS' if len(train) >= 200 else 'FAIL', '|',
      'requirement >=50 eval:', 'PASS' if len(eval_) >= 50 else 'FAIL')

raw 1595 -> clean 1591   (train pool 1191, test pool 400)
  dropped: exact/near duplicate                           2
  dropped: longer than 300 chars                          2
  dropped: train copy of a test text (leakage)            0
  repaired: whitespace                                    67

sampled: train 300, eval 120;  train/eval overlap (normalized text): 0
requirement >=200 train: PASS | requirement >=50 eval: PASS


In [4]:
# ---- statistics: token length, label balance, format checks ----
try:
    import tiktoken
    os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
    _enc = tiktoken.get_encoding('cl100k_base')
    tok_len = lambda s: len(_enc.encode(s))
    TOK_NAME = 'tiktoken cl100k_base'
except Exception as e:
    tok_len = lambda s: len(re.findall(r"\w+|[^\w\s]", s))
    TOK_NAME = f'regex word/punctuation count (tiktoken unavailable: {type(e).__name__})'
print('token counter:', TOK_NAME, '(an approximation -- Claude uses its own tokenizer)')

def describe(vals):
    s = sorted(vals); n = len(s)
    q = lambda p: s[min(n - 1, int(p * n))]
    return {'n': n, 'min': s[0], 'p25': q(.25), 'median': q(.5), 'p75': q(.75), 'p90': q(.9),
            'max': s[-1], 'mean': round(sum(s) / n, 1)}

TOKEN_STATS = {}
for name, ds in (('train', train), ('eval', eval_)):
    TOKEN_STATS[name] = describe([tok_len(r['text']) for r in ds])
    print(f'{name:5s} token length', TOKEN_STATS[name])
lens = [tok_len(r['text']) for r in train]
buckets = Counter((l // 5) * 5 for l in lens)
peak = max(buckets.values())
print('train token-length histogram (5-token buckets):')
for b in sorted(buckets):
    print(f'  {b:3d}-{b + 4:<3d} | ' + '#' * max(1, round(buckets[b] / peak * 40)) + f' {buckets[b]}')

print()
BALANCE = {}
for name, ds in (('train', train), ('eval', eval_)):
    c = Counter(r['label'] for r in ds)
    BALANCE[name] = dict(c)
    print(f'{name:5s} label balance (max/min = {max(c.values()) / min(c.values()):.2f}):', dict(sorted(c.items())))
for name, ds in (('train', train), ('eval', eval_)):
    print(f'{name:5s} by group:', dict(Counter(r['cluster'] for r in ds)))

print()
CHECKS = {
    'has text + label strings': lambda r: isinstance(r.get('text'), str) and isinstance(r.get('label'), str),
    'text non-empty': lambda r: bool(r['text'].strip()),
    'no leading/trailing whitespace': lambda r: r['text'] == r['text'].strip(),
    'no double spaces / newlines / tabs': lambda r: not re.search(r'\s{2,}|[\n\t]', r['text']),
    'label in my intent set': lambda r: r['label'] in VALID,
    f'length <= {MAX_CHARS} chars': lambda r: len(r['text']) <= MAX_CHARS,
}
FORMAT_CHECKS = {}
allrows = train + eval_
for cname, fn in CHECKS.items():
    ok = sum(1 for r in allrows if fn(r))
    FORMAT_CHECKS[cname] = f'{ok}/{len(allrows)}'
    print(f'format check: {cname:38s} {ok}/{len(allrows)}')
non_ascii = sum(1 for r in allrows if any(ord(ch) > 126 or ord(ch) < 32 for ch in r['text']))
FORMAT_CHECKS['tickets containing non-ASCII characters (informational)'] = f'{non_ascii}/{len(allrows)}'
print(f'info: tickets containing non-ASCII characters: {non_ascii}/{len(allrows)} (kept as is)')

token counter: tiktoken cl100k_base (an approximation -- Claude uses its own tokenizer)
train token length {'n': 300, 'min': 3, 'p25': 8, 'median': 10, 'p75': 14, 'p90': 24, 'max': 57, 'mean': 12.7}
eval  token length {'n': 120, 'min': 3, 'p25': 8, 'median': 11, 'p75': 14, 'p90': 25, 'max': 71, 'mean': 13.1}
train token-length histogram (5-token buckets):
    0-4   | ## 6
    5-9   | ######################################## 125
   10-14  | ################################ 100
   15-19  | ########## 30
   20-24  | #### 13
   25-29  | ## 6
   30-34  | #### 11
   35-39  | # 3
   40-44  | # 4
   50-54  | # 1
   55-59  | # 1

train label balance (max/min = 1.00): {'card_arrival': 30, 'card_delivery_estimate': 30, 'card_not_working': 30, 'contactless_not_working': 30, 'declined_card_payment': 30, 'declined_transfer': 30, 'pending_card_payment': 30, 'pending_top_up': 30, 'pending_transfer': 30, 'virtual_card_not_working': 30}
eval  label balance (max/min = 1.00): {'card_arrival': 12, 'card_de

In [5]:
# ---- save the dataset + quality report (commit these) ----
os.makedirs('data', exist_ok=True)
def dump(path, rows):
    with open(path, 'w') as f:
        for r in rows:
            f.write(json.dumps({k: r[k] for k in ('text', 'label', 'cluster')}, sort_keys=True) + '\n')
dump('data/banking77_train.jsonl', train)
dump('data/banking77_eval.jsonl', eval_)
FINGERPRINT = hashlib.sha256(open('data/banking77_train.jsonl', 'rb').read() + open('data/banking77_eval.jsonl', 'rb').read()).hexdigest()[:16]
QUALITY = {
    'source': 'PolyAI Banking77 (Casanueva et al., 2020), CC BY 4.0', 'source_url': BASE_URL,
    'raw_file_sha256_16': {'train.csv': HASH_TRAIN, 'test.csv': HASH_TEST},
    'intents_used (chosen by hand, not random)': CLUSTERS, 'seed': SEED,
    'raw_rows_for_my_intents': len(raw), 'clean': len(clean_records), 'train': len(train), 'eval': len(eval_),
    'cleaning': dict(CLEAN_REPORT), 'train_eval_text_overlap': len(overlap),
    'token_counter': TOK_NAME, 'token_stats': TOKEN_STATS, 'label_balance': BALANCE,
    'format_checks': FORMAT_CHECKS, 'dataset_fingerprint': FINGERPRINT,
}
with open('data/banking77_quality_report.json', 'w') as f:
    json.dump(QUALITY, f, indent=2)
print('wrote data/banking77_train.jsonl, data/banking77_eval.jsonl, data/banking77_quality_report.json')
print('dataset fingerprint:', FINGERPRINT)

wrote data/banking77_train.jsonl, data/banking77_eval.jsonl, data/banking77_quality_report.json
dataset fingerprint: 8a6a4c8407aecdc2


**Data-quality read-out.**

My 10 intents had 1,595 rows in Banking77: 1,195 in the train file and 400 in the test file (40 per
intent). The train counts varied a lot, from 35 (`contactless_not_working`) to 159
(`pending_card_payment`). Cleaning found real, small problems: 67 rows (about 4%) with stray
leading, trailing or double whitespace, which I repaired; 2 duplicates and 2 messages over 300
characters, which I dropped; and no train copies of test texts and no conflicting labels. That
left 1,591 rows (1,191 train pool, 400 test pool), from which I sampled 300 train and 120 eval
with zero overlap between them.

Messages are short: median 10 tokens in train and 11 in eval,
counted with tiktoken `cl100k_base`, which only approximates Claude's tokenizer. All six format
checks pass on all 420 messages, and none contain non-ASCII characters. Label balance is exactly
30 per intent in train and 12 in eval, but that is by construction, not how the data is
distributed. The dataset fingerprint is `8a6a4c8407aecdc2`.

Two things to keep in mind. The 10 intents were chosen by hand as look-alikes, and 30/12 per
intent is my sampling choice. And Banking77 contains many paraphrases of the same question: 20 of
the 120 eval messages have a training message at 0.95 similarity or higher (median top-1
similarity 0.87), which flatters any method that retrieves similar messages.


## Part 2: Adapt with dynamic few-shot (local selection)

Same retrieval stack as Week 5: `all-MiniLM-L6-v2` embeddings and a FAISS `IndexFlatIP` over
normalized vectors (inner product = cosine), where the "documents" are the training messages.
For each eval message, the 4 most similar training messages (with their intents) go into the
prompt, most similar first. Retrieval looks only at the message text, never at the eval label.
I also check how close eval messages are to training messages, because Banking77 contains many
near-paraphrases.


In [6]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
K = 4

def embed(texts):
    return np.asarray(emb.encode(texts, normalize_embeddings=True)).astype('float32')

train_vecs = embed([r['text'] for r in train])
eval_vecs = embed([r['text'] for r in eval_])
index = faiss.IndexFlatIP(train_vecs.shape[1]); index.add(train_vecs)

def retrieve(vecs, k=K):
    sims, ids = index.search(vecs, k)
    return [([int(j) for j in ids[i]], [float(s) for s in sims[i]]) for i in range(len(ids))]

RETR = retrieve(eval_vecs, K)          # one retrieval per eval message, reused by every later cell

def knn_vote(ids):
    """Plurality intent among retrieved neighbors; ties go to the most similar neighbor's intent."""
    votes = Counter(train[j]['label'] for j in ids)
    top = max(votes.values())
    tied = {l for l, v in votes.items() if v == top}
    for j in ids:
        if train[j]['label'] in tied:
            return train[j]['label']

def relation(i):
    """How the most similar training example relates to the eval message's intent."""
    lab = train[RETR[i][0][0]]['label']
    gold = eval_[i]['label']
    if lab == gold:
        return 'top-1 is the same intent'
    return 'top-1 is a sibling intent' if CLUSTER_OF[lab] == CLUSTER_OF[gold] else 'top-1 is from another group'

# one example of what the model will see: the first eval message whose top-1 neighbor is a sibling intent
ex_i = next((i for i in range(len(eval_)) if relation(i) == 'top-1 is a sibling intent'), 0)
print('eval message:', eval_[ex_i]['text'], '(gold:', eval_[ex_i]['label'] + ')')
print('retrieved few-shot examples:')
for j, s in zip(*RETR[ex_i]):
    print(f'   {s:.2f}  {train[j]["text"]}  ->  {train[j]["label"]}')

top1 = sorted(s[0] for _, s in RETR)
print(f'\ntop-1 train similarity per eval message: median {top1[len(top1) // 2]:.2f}, min {top1[0]:.2f}, '
      f'max {top1[-1]:.2f}, share >= 0.90: {sum(s >= 0.90 for s in top1) / len(top1):.0%}')
near = sorted(((s[0], i) for i, (_, s) in enumerate(RETR) if s[0] >= 0.95), reverse=True)
print(f'eval messages with a training message at similarity >= 0.95 (near-paraphrases): {len(near)}/{len(eval_)}')
for s, i in near[:4]:
    print(f'   {s:.2f}  eval: {eval_[i]["text"]!r}  vs  train: {train[RETR[i][0][0]]["text"]!r}')
print('how the most similar training example relates to the eval intent:', dict(Counter(relation(i) for i in range(len(eval_)))))
pure = sum(all(train[j]['label'] == eval_[i]['label'] for j in RETR[i][0]) for i in range(len(eval_)))
print(f'eval messages whose {K} neighbors ALL carry the gold intent: {pure}/{len(eval_)}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

eval message: The card got declined twice when I tried to use it to buy something online yesterday. (gold: declined_transfer)
retrieved few-shot examples:
   0.76  When I use my card, it says decline?  ->  declined_card_payment
   0.74  Why was my payment declined? I keep trying my card but it's not working.  ->  declined_card_payment
   0.74  Why do you keep declining my payment?I tried several times with this card and it is just not working.  ->  declined_card_payment
   0.73  Tell me why my card payment has been declined.  ->  declined_card_payment

top-1 train similarity per eval message: median 0.87, min 0.55, max 0.99, share >= 0.90: 34%
eval messages with a training message at similarity >= 0.95 (near-paraphrases): 20/120
   0.99  eval: "Why isn't my contactless not working?"  vs  train: 'Why is my contactless not working?'
   0.98  eval: 'Why was my Payment declined'  vs  train: 'Why did my payment get declined?'
   0.98  eval: 'I would like to know why my transfer was declined

## Part 3: Evaluate, including one failure

Zero-shot and dynamic few-shot use the identical instruction line; the only difference is the 4
retrieved examples. The zero-shot prompt shows only the 10 intent names (no definitions). Two extra
rows cost no model call and anchor the comparison for the memo: a kNN vote (predict the plurality
intent of the same 4 neighbors, no LLM at all) and the majority-class floor. Predictions that
don't parse to a valid intent count as wrong and are reported separately, never dropped.


In [7]:
import getpass

def _get_api_key():
    """Colab secret, then env var, then an in-notebook prompt (same pattern as Weeks 5-7)."""
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY')
    if key:
        return key
    try:
        return getpass.getpass('Enter your ANTHROPIC_API_KEY: ')
    except Exception:
        return None

MODEL = 'claude-haiku-4-5-20251001'
_api_key = _get_api_key()
_client = None
if _api_key:
    import anthropic
    _client = anthropic.Anthropic(api_key=_api_key, timeout=30.0, max_retries=1)
    print('API key found; client ready (30s timeout per call).')
else:
    print('[API-BLOCKED] no key found -- the dataset and retrieval above are real; the scored eval is skipped.')

def build_prompt(text, shots=None):
    head = ('Classify the customer message into exactly one of these intents: ' + ', '.join(LABELS)
            + '.\nReply with only the intent.\n\n')
    body = ''.join(f'Message: {s["text"]}\nIntent: {s["label"]}\n\n' for s in (shots or []))
    return head + body + f'Message: {text}\nIntent:'

def parse_label(raw):
    """-> (label, was_exact). 'invalid' if no valid intent can be found in the reply."""
    if raw is None:
        return 'invalid', False
    s = raw.strip().lower()
    s = re.sub(r'^(intent|label|answer)\s*[:\-]\s*', '', s).strip(' .`*"\'\n')
    if s in VALID:
        return s, True
    hits = sorted((m.start(), lab) for lab in LABELS for m in re.finditer(rf'\b{lab}\b', raw.lower()))
    if hits:
        return hits[0][1], False
    return 'invalid', False

TEMP_PINNED = True      # flips to False (with a printed note) if the API refuses temperature for this model

def call_claude(prompt, retries=3):
    global TEMP_PINNED
    last = None
    for attempt in range(retries):
        try:
            t0 = time.time()
            kwargs = dict(model=MODEL, max_tokens=24, messages=[{'role': 'user', 'content': prompt}])
            if TEMP_PINNED:
                kwargs['extra_body'] = {'temperature': 0}
            resp = _client.messages.create(**kwargs)
            return {'raw': resp.content[0].text, 'in_tok': resp.usage.input_tokens,
                    'out_tok': resp.usage.output_tokens, 'secs': time.time() - t0, 'error': None}
        except Exception as e:
            last = f'{type(e).__name__}: {e}'
            if TEMP_PINNED and getattr(e, 'status_code', None) == 400 and 'temperature' in str(e).lower():
                TEMP_PINNED = False
                print('NOTE: the API refused temperature for this model -- continuing WITHOUT pinning it. '
                      'Outputs may vary slightly between runs; say so in the write-up.')
                continue
            if getattr(e, 'status_code', None) in (400, 401, 403, 404):
                break
            time.sleep(2 ** attempt)
    return {'raw': None, 'in_tok': 0, 'out_tok': 0, 'secs': 0.0, 'error': last}

CALLS = {}      # (variant, message) -> result; every pair is called exactly once
def classify(variant, text, shots=None, retries=3):
    key = (variant, text)
    if key not in CALLS or CALLS[key]['error']:      # successes are cached; failures are retried on re-run
        out = call_claude(build_prompt(text, shots), retries=retries)
        out['pred'], out['exact'] = parse_label(out['raw'])
        CALLS[key] = out
    return CALLS[key]


API key found; client ready (30s timeout per call).


In [8]:
# Step 1: ONE test call first, so a bad key / model name shows up in seconds, not minutes.
API_OK = False
if _client is not None:
    t0 = time.time()
    probe = classify('zero', eval_[0]['text'], retries=2)     # cached, so the real run reuses it
    if probe['error']:
        print(f'[API-BLOCKED] test call failed after {time.time() - t0:.1f}s:\n  {probe["error"]}')
        print('Fix the key / model name / network above, then re-run this cell. Nothing else will be called.')
    else:
        API_OK = True
        print(f'test call OK in {time.time() - t0:.1f}s -> raw reply {probe["raw"]!r}, parsed {parse_label(probe["raw"])}')
        print('temperature pinned to 0:', TEMP_PINNED)

test call OK in 0.9s -> raw reply 'card_delivery_estimate', parsed ('card_delivery_estimate', True)
temperature pinned to 0: True


In [9]:
# Step 2: the real run -- 2 calls per eval message. Prints a line every 8 messages.
ROWS = []
if API_OK:
    t_start = time.time()
    bad_streak = 0
    for i, r in enumerate(eval_):
        if i % 8 == 0:
            n_err = sum(1 for c in CALLS.values() if c['error'])
            print(f'  message {i}/{len(eval_)}   {time.time() - t_start:.0f}s elapsed   {n_err} failed calls so far', flush=True)
        ids, sims = RETR[i]
        shots = [train[j] for j in ids]
        z = classify('zero', r['text'])
        f = classify('few', r['text'], shots)
        bad_streak = bad_streak + 1 if (z['error'] or f['error']) else 0
        if bad_streak >= 3:
            print(f'stopping: 3 messages in a row failed. Last error: {z["error"] or f["error"]}')
            break
        if z['error'] or f['error']:
            continue                       # never score an API failure as a wrong answer
        ROWS.append({
            'i': i, 'text': r['text'], 'gold': r['label'], 'cluster': r['cluster'],
            'zero': z['pred'], 'few': f['pred'], 'knn': knn_vote(ids), 'rel': relation(i),
            'shot_labels': [train[j]['label'] for j in ids], 'shot_texts': [train[j]['text'] for j in ids],
            'top1_sim': sims[0],
        })
    errs = sum(1 for c in CALLS.values() if c['error'])
    first = next((c['error'] for c in CALLS.values() if c['error']), None)
    if first:
        print('first API error:', first)
    print(f'{len(CALLS)} model calls made ({len(ROWS)} eval messages x 2 variants), {errs} failed after retries')
    if len(ROWS) < len(eval_):
        print(f'WARNING: only {len(ROWS)}/{len(eval_)} messages scored -- re-run this cell to retry the failed ones before reading any numbers.')
else:
    print('no scored run (see the test-call message above)')

  message 0/120   0s elapsed   0 failed calls so far
  message 8/120   8s elapsed   0 failed calls so far
  message 16/120   17s elapsed   0 failed calls so far
  message 24/120   26s elapsed   0 failed calls so far
  message 32/120   34s elapsed   0 failed calls so far
  message 40/120   42s elapsed   0 failed calls so far
  message 48/120   51s elapsed   0 failed calls so far
  message 56/120   59s elapsed   0 failed calls so far
  message 64/120   68s elapsed   0 failed calls so far
  message 72/120   76s elapsed   0 failed calls so far
  message 80/120   85s elapsed   0 failed calls so far
  message 88/120   93s elapsed   0 failed calls so far
  message 96/120   102s elapsed   0 failed calls so far
  message 104/120   110s elapsed   0 failed calls so far
  message 112/120   119s elapsed   0 failed calls so far
240 model calls made (120 eval messages x 2 variants), 0 failed after retries


In [10]:
def accuracy(preds, golds):
    return sum(p == g for p, g in zip(preds, golds)) / len(golds)

def macro_f1(preds, golds):
    f1s = []
    for lab in LABELS:
        tp = sum(p == lab and g == lab for p, g in zip(preds, golds))
        fp = sum(p == lab and g != lab for p, g in zip(preds, golds))
        fn = sum(p != lab and g == lab for p, g in zip(preds, golds))
        f1s.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0)
    return sum(f1s) / len(f1s)

def wilson(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n."""
    if n == 0:
        return 0.0, 0.0
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n)
    a = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return (c - a) / d, (c + a) / d

def sign_test_p(b, c):
    """Two-sided exact sign test on the disagreements (b = few-shot only right, c = zero-shot only right)."""
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)

if ROWS:
    gold = [r['gold'] for r in ROWS]
    majority = Counter(r['label'] for r in train).most_common(1)[0][0]
    table = [('zero-shot', [r['zero'] for r in ROWS]), (f'dynamic {K}-shot', [r['few'] for r in ROWS]),
             (f'kNN vote of the same {K} (no LLM)', [r['knn'] for r in ROWS]),
             (f'majority class ({majority})', [majority] * len(ROWS))]
    print(f'{"variant":36s} accuracy [95% CI]            macro-F1')
    for name, preds in table:
        k = sum(p == g for p, g in zip(preds, gold)); lo, hi = wilson(k, len(gold))
        print(f'{name:36s} {k / len(gold):6.1%} [{lo:5.1%}, {hi:5.1%}]     {macro_f1(preds, gold):.3f}')

    print()
    for v, label in (('zero', 'zero-shot'), ('few', f'dynamic {K}-shot')):
        cs = [c for (vv, _), c in CALLS.items() if vv == v]
        inv = sum(1 for c in cs if c['pred'] == 'invalid')
        inexact = sum(1 for c in cs if c['pred'] != 'invalid' and not c['exact'])
        print(f'{label:16s} avg input tokens {sum(c["in_tok"] for c in cs) / len(cs):6.1f}   '
              f'avg output tokens {sum(c["out_tok"] for c in cs) / len(cs):4.1f}   '
              f'avg latency {sum(c["secs"] for c in cs) / len(cs):.2f}s   '
              f'unparseable {inv}   parsed-but-not-exact {inexact}   api errors {sum(1 for c in cs if c["error"])}')

    both = sum(r['zero'] == r['gold'] and r['few'] == r['gold'] for r in ROWS)
    few_only = sum(r['zero'] != r['gold'] and r['few'] == r['gold'] for r in ROWS)
    zero_only = sum(r['zero'] == r['gold'] and r['few'] != r['gold'] for r in ROWS)
    neither = len(ROWS) - both - few_only - zero_only
    print(f'\npaired: both right {both}, only few-shot right {few_only}, only zero-shot right {zero_only}, both wrong {neither}')
    print(f'exact sign test on the {few_only + zero_only} disagreements: p = {sign_test_p(few_only, zero_only):.3f}  (n = {len(ROWS)})')

    print('\nmost common mistakes (gold -> predicted):')
    for v, label in (('zero', 'zero-shot'), ('few', f'dynamic {K}-shot')):
        conf = Counter((r['gold'], r[v]) for r in ROWS if r[v] != r['gold']).most_common(6)
        print(f'  {label}: ' + ('; '.join(f'{g} -> {p} x{n}' for (g, p), n in conf) if conf else 'none'))
else:
    print('skipped (no model calls were made)')

variant                              accuracy [95% CI]            macro-F1
zero-shot                             85.8% [78.5%, 91.0%]     0.846
dynamic 4-shot                        91.7% [85.3%, 95.4%]     0.915
kNN vote of the same 4 (no LLM)       95.8% [90.6%, 98.2%]     0.958
majority class (declined_transfer)    10.0% [ 5.8%, 16.7%]     0.018

zero-shot        avg input tokens  103.3   avg output tokens  7.9   avg latency 0.54s   unparseable 0   parsed-but-not-exact 0   api errors 0
dynamic 4-shot   avg input tokens  193.8   avg output tokens  7.7   avg latency 0.53s   unparseable 0   parsed-but-not-exact 0   api errors 0

paired: both right 101, only few-shot right 9, only zero-shot right 2, both wrong 8
exact sign test on the 11 disagreements: p = 0.065  (n = 120)

most common mistakes (gold -> predicted):
  zero-shot: card_arrival -> card_delivery_estimate x9; declined_transfer -> declined_card_payment x4; pending_card_payment -> declined_card_payment x1; contactless_not_worki

In [11]:
def slice_report(title, keyfn):
    print(title)
    print(f'  {"slice":34s} {"n":>3s}  {"zero":>6s}  {"few":>6s}  {"knn":>6s}')
    groups = defaultdict(list)
    for r in ROWS:
        groups[keyfn(r)].append(r)
    for name in sorted(groups):
        g = groups[name]
        z, f, k = (sum(r[v] == r['gold'] for r in g) / len(g) for v in ('zero', 'few', 'knn'))
        note = '   <-- few-shot underperforms' if f < z else ''
        small = '  (small n)' if len(g) < 15 else ''
        print(f'  {str(name):34s} {len(g):3d}  {z:6.0%}  {f:6.0%}  {k:6.0%}{note}{small}')
    print()

if ROWS:
    slice_report('by group of look-alike intents:', lambda r: r['cluster'])
    slice_report('by intent:', lambda r: r['gold'])
    slice_report('by how the most similar training example relates to the answer:', lambda r: r['rel'])
    slice_report('by retrieval purity:', lambda r: f'all {K} neighbors = gold' if all(l == r['gold'] for l in r['shot_labels']) else 'neighbors mixed/wrong')

    for title, cond in [('few-shot wrong where zero-shot was right', lambda r: r['few'] != r['gold'] == r['zero']),
                        ('few-shot right, zero-shot wrong', lambda r: r['few'] == r['gold'] != r['zero'])]:
        hits = [r for r in ROWS if cond(r)]
        print(f'--- {title}: {len(hits)} ---')
        for r in hits[:12]:
            print(f'message: {r["text"]!r}')
            print(f'   gold={r["gold"]}  zero={r["zero"]}  few={r["few"]}  knn={r["knn"]}')
            for t_, l_ in zip(r['shot_texts'], r['shot_labels']):
                print(f'     shot: {t_!r} -> {l_}')
else:
    print('skipped (no model calls were made)')

by group of look-alike intents:
  slice                                n    zero     few     knn
  declined                            24     79%     83%     88%
  delivery                            24     62%     92%     96%
  not_working                         36     94%     89%     97%   <-- few-shot underperforms
  pending                             36     97%    100%    100%

by intent:
  slice                                n    zero     few     knn
  card_arrival                        12     25%     83%     92%  (small n)
  card_delivery_estimate              12    100%    100%    100%  (small n)
  card_not_working                    12    100%    100%    100%  (small n)
  contactless_not_working             12     92%     83%    100%   <-- few-shot underperforms  (small n)
  declined_card_payment               12    100%    100%    100%  (small n)
  declined_transfer                   12     58%     67%     75%  (small n)
  pending_card_payment                12     92%    

### Extra experiment: give the prompts a one-line definition per intent

Most of few-shot's gain came from one intent (`card_arrival`), where the zero-shot prompt only has
the intent names to go on. The obvious cheaper fix is to say what each intent means. This
experiment adds a one-line definition for each of the 10 intents to the prompt and re-runs two
variants: zero-shot with definitions, and few-shot with definitions (the same 4 retrieved examples
as before). Nothing else changes.

In [12]:
# ---- extra experiment: one-line definitions in the prompt (about 240 calls) ----
DEFINITIONS = {
    'card_arrival': "the customer's ordered card has not arrived yet, or they ask where it is",
    'card_delivery_estimate': "the customer asks how long card delivery takes or when it should arrive",
    'card_not_working': "the customer's physical card is not working",
    'virtual_card_not_working': "the customer's virtual or disposable card is not working",
    'contactless_not_working': "contactless payments are not working",
    'pending_card_payment': "a card payment is still pending or has not gone through yet",
    'pending_transfer': "a bank transfer is still pending",
    'pending_top_up': "a top-up to the account is still pending",
    'declined_card_payment': "a card payment was declined",
    'declined_transfer': "a bank transfer was declined",
}
assert set(DEFINITIONS) == set(LABELS)

def build_prompt_defs(text, shots=None):
    head = ('Classify the customer message into exactly one of these intents (what each one means is given):\n'
            + '\n'.join(f'- {l}: {DEFINITIONS[l]}' for l in LABELS)
            + '\nReply with only the intent name.\n\n')
    body = ''.join(f'Message: {s["text"]}\nIntent: {s["label"]}\n\n' for s in (shots or []))
    return head + body + f'Message: {text}\nIntent:'

def classify_defs(variant, text, shots=None):
    key = (variant, text)
    if key not in CALLS or CALLS[key]['error']:
        out = call_claude(build_prompt_defs(text, shots))
        out['pred'], out['exact'] = parse_label(out['raw'])
        CALLS[key] = out
    return CALLS[key]

DEF_ROWS = []
if API_OK and ROWS:
    t_start = time.time()
    for n_done, r in enumerate(ROWS):
        if n_done % 20 == 0:
            print(f'  message {n_done}/{len(ROWS)}   {time.time() - t_start:.0f}s elapsed', flush=True)
        ids, _ = RETR[r['i']]
        shots = [train[j] for j in ids]
        zd = classify_defs('zero_def', r['text'])
        fd = classify_defs('few_def', r['text'], shots)
        if zd['error'] or fd['error']:
            continue
        DEF_ROWS.append({**r, 'zero_def': zd['pred'], 'few_def': fd['pred']})
    if len(DEF_ROWS) < len(ROWS):
        print(f'WARNING: only {len(DEF_ROWS)}/{len(ROWS)} messages scored -- re-run this cell before reading numbers.')
    print('done:', len(DEF_ROWS), 'messages')
else:
    print('skipped (no API access or no scored messages)')

  message 0/120   0s elapsed
  message 20/120   21s elapsed
  message 40/120   43s elapsed
  message 60/120   63s elapsed
  message 80/120   85s elapsed
  message 100/120   107s elapsed
done: 120 messages


In [13]:
# ---- results of the definitions experiment ----
if DEF_ROWS:
    g = [r['gold'] for r in DEF_ROWS]
    variants = [('zero', 'zero-shot (names only)'), ('zero_def', 'zero-shot + definitions'),
                ('few', 'dynamic 4-shot (names only)'), ('few_def', 'dynamic 4-shot + definitions'),
                ('knn', 'kNN vote (no LLM)')]
    print(f'{"variant":32s} accuracy [95% CI]            macro-F1   input tok')
    for key, name in variants:
        preds = [r[key] for r in DEF_ROWS]
        k = sum(p == t for p, t in zip(preds, g)); lo, hi = wilson(k, len(g))
        cs = [c for (vv, _), c in CALLS.items() if vv == key]
        tok = f'{sum(c["in_tok"] for c in cs) / len(cs):9.1f}' if cs else '     none'
        print(f'{name:32s} {k / len(g):6.1%} [{lo:5.1%}, {hi:5.1%}]     {macro_f1(preds, g):.3f}   {tok}')

    def pair(a, b, label):
        a_only = sum(r[a] == r['gold'] != r[b] for r in DEF_ROWS)
        b_only = sum(r[b] == r['gold'] != r[a] for r in DEF_ROWS)
        print(f'  {label}: only {a} right {a_only}, only {b} right {b_only}, sign test p = {sign_test_p(a_only, b_only):.3f}')
    print('\npaired comparisons:')
    pair('zero_def', 'zero', 'definitions help zero-shot?')
    pair('few', 'zero_def', 'is few-shot still ahead of zero-shot + definitions?')
    pair('few_def', 'few', 'do definitions help few-shot?')

    print('\nby intent (accuracy):')
    print(f'  {"intent":26s} {"n":>3s} {"zero":>6s} {"+defs":>6s} {"few":>6s} {"+defs":>6s} {"knn":>6s}')
    for lab in LABELS:
        gg = [r for r in DEF_ROWS if r['gold'] == lab]
        a = [sum(r[v] == lab for r in gg) / len(gg) for v in ('zero', 'zero_def', 'few', 'few_def', 'knn')]
        print(f'  {lab:26s} {len(gg):3d} ' + ' '.join(f'{x:6.0%}' for x in a))

    print('\nmost common mistakes (gold -> predicted):')
    for key, name in (('zero_def', 'zero-shot + definitions'), ('few_def', 'few-shot + definitions')):
        conf = Counter((r['gold'], r[key]) for r in DEF_ROWS if r[key] != r['gold']).most_common(6)
        print(f'  {name}: ' + ('; '.join(f'{a} -> {b} x{n}' for (a, b), n in conf) if conf else 'none'))

    for title, cond in [('zero-shot + definitions wrong, names-only few-shot right', lambda r: r['zero_def'] != r['gold'] == r['few']),
                        ('few-shot + definitions wrong, zero-shot + definitions right', lambda r: r['few_def'] != r['gold'] == r['zero_def'])]:
        hits = [r for r in DEF_ROWS if cond(r)]
        print(f'\n--- {title}: {len(hits)} ---')
        for r in hits[:8]:
            print(f'message: {r["text"]!r}')
            print(f'   gold={r["gold"]}  zero={r["zero"]}  zero+defs={r["zero_def"]}  few={r["few"]}  few+defs={r["few_def"]}')
else:
    print('skipped')

variant                          accuracy [95% CI]            macro-F1   input tok
zero-shot (names only)            85.8% [78.5%, 91.0%]     0.846       103.3
zero-shot + definitions           90.0% [83.3%, 94.2%]     0.916       223.3
dynamic 4-shot (names only)       91.7% [85.3%, 95.4%]     0.915       193.8
dynamic 4-shot + definitions      93.3% [87.4%, 96.6%]     0.940       313.8
kNN vote (no LLM)                 95.8% [90.6%, 98.2%]     0.958        none

paired comparisons:
  definitions help zero-shot?: only zero_def right 9, only zero right 4, sign test p = 0.267
  is few-shot still ahead of zero-shot + definitions?: only few right 5, only zero_def right 3, sign test p = 0.727
  do definitions help few-shot?: only few_def right 3, only few right 1, sign test p = 0.625

by intent (accuracy):
  intent                       n   zero  +defs    few  +defs    knn
  card_arrival                12    25%    92%    83%   100%    92%
  card_delivery_estimate      12   100%    92%   1

In [14]:
# ---- diagnostic: how many replies could not be parsed, and what did the model say instead? ----
for v, name in (('zero', 'zero-shot'), ('zero_def', 'zero-shot + definitions'),
                ('few', 'dynamic 4-shot'), ('few_def', 'dynamic 4-shot + definitions')):
    cs = [(t, c) for (vv, t), c in CALLS.items() if vv == v]
    inv = [t for t, c in cs if c['pred'] == 'invalid']
    inexact = sum(1 for t, c in cs if c['pred'] != 'invalid' and not c['exact'])
    print(f'{name:30s} unparseable {len(inv):2d}/{len(cs)}   parsed-but-not-exact {inexact}')
print()
for v in ('zero_def', 'few_def'):
    for t, c in [(t, c) for (vv, t), c in CALLS.items() if vv == v and c['pred'] == 'invalid'][:8]:
        print(f'{v:9s} message: {t!r}')
        print(f'          raw reply: {c["raw"]!r}')

zero-shot                      unparseable  0/120   parsed-but-not-exact 0
zero-shot + definitions        unparseable  5/120   parsed-but-not-exact 0
dynamic 4-shot                 unparseable  0/120   parsed-but-not-exact 0
dynamic 4-shot + definitions   unparseable  2/120   parsed-but-not-exact 0

zero_def  message: 'How do i finalize a card payment'
          raw reply: 'I cannot classify this message into any of the provided intents. The customer is asking how to complete a card payment ('
zero_def  message: 'Can I make multiple online transactions with my virtual card?'
          raw reply: "This message doesn't fit any of the provided intents. The customer is asking a general question about the functionality of their"
zero_def  message: 'what is the charges for new card?'
          raw reply: "This message doesn't fit any of the provided intents. The customer is asking about charges/fees for a new card"
zero_def  message: 'what does pending mean?'
          raw reply: 'I cannot c

## Results

| | accuracy [95% CI] | macro-F1 | input tokens/msg |
|---|---|---|---|
| zero-shot (names only) | 85.8% [78.5, 91.0] | 0.846 | 103 |
| zero-shot + definitions* | 90.0% [83.3, 94.2] | 0.916 | 223 |
| dynamic 4-shot (names only) | 91.7% [85.3, 95.4] | 0.915 | 194 |
| dynamic 4-shot + definitions* | 93.3% [87.4, 96.6] | 0.940 | 314 |
| kNN vote of the same 4 (no LLM) | 95.8% [90.6, 98.2] | 0.958 | none |
| majority class | 10.0% [5.8, 16.7] | 0.018 | none |

**Few-shot vs zero-shot.** Of 120 messages, 101 were right with both prompts, 9 only with few-shot, 2 only with zero-shot, and 8 with neither. The sign test on those 11 disagreements gives p = 0.065: suggestive, not conclusive, and the intervals overlap. Few-shot adds about 90 input tokens per message and is no slower (0.51s each).

**Where the gain came from.** 7 of the 9 few-shot-only wins are in one intent, `card_arrival`, mostly "what's the tracking number for my card" messages. Zero-shot sent those to `card_delivery_estimate` and scored 25% on `card_arrival`, against 83% with few-shot. With only intent names, the model can't know that this dataset files tracking questions under `card_arrival`. The retrieved examples tell it, so the gain is mostly about this dataset's labeling convention, not general knowledge. A one-line definition lifted zero-shot on that intent to 92%, which supports this.

**The slice where few-shot underperformed.** In the `not_working` group (36 messages), zero-shot got 34 right and few-shot 32. The two misses:

- "Why did the disposable virtual card which I used to pay a gym subscription get denied?" (gold `virtual_card_not_working`): few-shot said `declined_card_payment`.
- "How can I find out where my contactless works?" (gold `contactless_not_working`): few-shot said `card_delivery_estimate`.

In both, all 4 retrieved examples carried the correct intent, and the kNN vote was right, so the model went against correct examples. I can't say why. The "denied" wording may pull toward `declined_card_payment`, but I didn't test that, and I can't explain the second at all. The contactless message is also wrong in both few-shot variants and right in both zero-shot ones, which at temperature 0 points to a stable effect, not sampling noise. It is still tiny: two messages out of 36, within noise, with few-shot ahead overall.

**Definitions don't separate the model variants.** Zero-shot + definitions beat names-only zero-shot on 9 messages and lost on 4 (p = 0.27). Names-only few-shot vs zero-shot + definitions was 5 vs 3 (p = 0.73), and definitions on top of few-shot was 3 vs 1 (p = 0.63). With 120 messages I can't rank these. Definitions are also not cheaper: the 10 definitions add about 120 tokens per message, more than the roughly 90 that 4 retrieved examples add. What they save is infrastructure (no labeled set, embedder or index).

**The model sometimes declines.** The definition prompts returned a non-intent reply on 5 of 120 messages (zero-shot) and 2 of 120 (few-shot), against 0 for the names-only runs. Six of those seven say outright that the message fits none of the 10 intents; the seventh was cut off by my 24-token limit. I count them as wrong, which costs the definition variants some points. In a real router, "doesn't fit" is a useful answer, because it can send the message to a person. Among the messages they did classify, accuracy was 108/115 (94%) and 112/118 (95%), but those are different message sets, so I'm not claiming that as a result.

**Where everything struggles.** When the single most similar training message came from a sibling intent (6 messages), all three methods got 17%. `declined_transfer` was weak everywhere (58% zero-shot, 67% few-shot, 75% kNN, and 58% with definitions). One message I saw, "The card got declined twice when I tried to use it to buy something online yesterday," describes a card payment but is labeled `declined_transfer`, so some of these errors look like ambiguous labels, not model mistakes. That is one example, not an audit.

**The no-LLM baseline wins.** The kNN vote had the highest accuracy, 95.8%, with no model call, though its interval overlaps the others. Among the 96 messages whose 4 neighbors were all correct, kNN got 100%, few-shot 97% and zero-shot 91%. It is probably helped by the near-paraphrases in this dataset.

**Not tested:** a larger example pool, a different K, or another model. This is one run on 10 hand-picked intents with 12 messages each.

## Part 4: Decision memo

**Recommendation: don't fine-tune. Use retrieval, with the no-LLM kNN vote as the first line. Dynamic
few-shot, or zero-shot with one-line intent definitions, is the next step up; the two are
statistically tied. Names-only zero-shot is the one to avoid.** The first of those
rests on a favorable dataset (see the last paragraph).

| option | accuracy / macro-F1 | cost per message | what it needs to keep working |
|---|---|---|---|
| do nothing (majority class, or people route) | 10.0% / 0.018 | people reading every message | staff time |
| prompt only (zero-shot) | 85.8% / 0.846 | 103.3 input tokens, 0.51s | just the prompt |
| retrieval only (kNN vote, no LLM) | 95.8% / 0.958 | one embedding lookup, no model call | labeled set + embedder + index |
| zero-shot + intent definitions | 90.0% / 0.916 | 223.3 input tokens | just the prompt, kept in sync with the intent list |
| dynamic few-shot | 91.7% / 0.915 | 193.8 input tokens, 0.51s | labeled set + embedder + index |
| dynamic few-shot + definitions | 93.3% / 0.940 | 313.8 input tokens | all of the above |
| fine-tuning | not run | training + hosting | labeled set, training pipeline, retraining when intents change |

**Why not names-only zero-shot.** It was the weakest at 85.8%, and 25% on `card_arrival`, because
intent names alone don't say how this dataset files tracking questions. A one-line definition per
intent lifted it to 90.0% (92% on `card_arrival`), and few-shot with 4 examples reached 91.7%. The
two are not distinguishable here (5 vs 3 on the paired test, p = 0.73). Definitions cost more input
tokens than the examples (223 vs 194); what they save is the labeled set, embedder and index. I
wrote those definitions after seeing the first results, so 90.0% is optimistic, and the definition
prompts made the model decline to answer on 7 of 240 calls (it said the message fit none of
the intents), which count as wrong here but would be useful in a real router, since those messages
could go to a person.

**Why retrieval first.** The kNN vote was the most accurate at 95.8% and needs no model call, so no
per-message API cost and no latency. Few-shot's one real weakness showed up here: it overrode
correct examples on 2 messages. The cost is that it needs a maintained labeled set, an embedder and
an index. The intervals overlap, so I'd not claim kNN beats few-shot, only that it is at least as
good for this data at far lower cost. Combining them (use the vote when the neighbors agree, and
the model only when they split) is an idea I did not test.

**When fine-tuning would be worth it.** Nothing here points to it yet. I'd revisit if:
- *the remaining error is model error, not label noise.* The best method still misses 4% here, and
  some of those misses (`declined_transfer`) look like ambiguous labels, which fine-tuning cannot
  fix. It is only worth paying for if a real audit shows the misses are the model's.
- *volume makes tokens the bill.* Few-shot adds about 90 input tokens per message and definitions about 120. Fine-tuning pays
  off only if training + hosting + retraining costs less than the tokens a shorter prompt saves at
  my volume. I haven't priced it and don't know the volume.
- *the intent list is stable and large.* I used 10 of 77 intents; retrieval handles a changing list
  by editing data, while a fine-tuned model must be retrained.
- *I can't send messages to an API,* for privacy or latency, and need a small self-hosted model.
  That is a constraint, not a result from this run.

**How much to trust this.** Banking77 contains many paraphrases of the same question, which favors
retrieval, and I picked the intents. Before deciding for real I'd test on messages phrased unlike
anything in the training set. The sample is small too: 12 messages per intent, so one message
moves an intent by 8 points.
